## Splitting and ingesting the content of various URLs (across UK destinations)

In [ ]:
# Preparing the Chroma DB collections

from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings

ollama_embeddings = OllamaEmbeddings(
    model="bge-m3",
    keep_alive=1800,  # 30 minutes
)

uk_granular_collection = Chroma(
    collection_name="uk_granular",
    embedding_function=ollama_embeddings,
)

uk_granular_collection.reset_collection() #A

In [ ]:
# Splitting and ingesting HTML content with the HTMLSectionSplitter

from langchain_text_splitters import HTMLSectionSplitter
from langchain_community.document_loaders import AsyncHtmlLoader


uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall",
]

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]
headers_to_split_on = [("h1", "Header 1"),("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(headers_to_split_on=headers_to_split_on)


def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content #B
        temp_chunks = html_section_splitter.split_text(
            html_string) #C
        h2_temp_chunks = [chunk for chunk in 
                          temp_chunks if "Header 2" 
                          in chunk.metadata] #D
        all_chunks.extend(h2_temp_chunks) 

    return all_chunks


for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(
        destination_url) #E
    docs =  html_loader.load() #F
    
    for doc in docs:
        print(doc.metadata)
        granular_chunks = split_docs_into_granular_chunks(docs)
        uk_granular_collection.add_documents(
            documents=granular_chunks)

#A In case it exists
#B Extract the HTML text from the document
#C Each chunk is a H1 or H2 HTML section
#D Only keep content associated with H2 sections        
#E Loader for one destination
#F Documents of one destination

## Rewrite-retrieve-read

In [ ]:
# Retrieving content with original user question

user_question = "Tell me some fun things I can enjoy in Cornwall"
initial_results = uk_granular_collection.similarity_search(query=user_question,k=4)
for doc in initial_results:
    print(doc)

In [ ]:
# Question rewrite

# Setting up the query rewriter chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


llm = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=8192, # overrides Ollama’s default context for this model invocation.
    num_predict=192, # limits summary generation.
    temperature=0,
    reasoning=False,
    # keep_alive=1800, # keeps the model loaded for 30 minutes.
)

rewriter_prompt_template = """
Generate search query for the Chroma DB vector store
from a user question, allowing for a more accurate 
response through semantic search.
Just return the revised Chroma DB query, with quotes around it. 

User question: {user_question}
Revised Chroma DB query:
"""

rewriter_prompt = ChatPromptTemplate.from_template(
    rewriter_prompt_template) 
rewriter_chain = rewriter_prompt | llm | StrOutputParser()

# Retrieving content with the rewritten query
user_question ="Tell me some fun things I can do in Cornwall"

search_query = rewriter_chain.invoke({"user_question": user_question})
print(search_query)

improved_results = uk_granular_collection.similarity_search(query=search_query,k=3)
for doc in improved_results:
    print(doc)